# RHOAI Release Confidence Classifier

**Goal:** Given a RHOAI feature at planning freeze, predict the probability it ships in its committed phase (EA1, EA2, or GA).

**Output:** Per-feature shipping confidence (0–100%), baked into the release planner demo at https://github.com/yuvalluria/rhoai-release-planner

This notebook is an **end-to-end record** of all experiments — including failed attempts and why they were abandoned — so the team has full context for every model decision.

## Model Version History

| Version | Training data | n | Slipped | Algorithm | AUC | Key finding |
|---|---|---|---|---|---|---|
| v1 | 3.5 only | 115 | 11 | RF + ROS | 91.9% | 4-way balancing experiment; ROS wins |
| v2 | 3.4 + 3.5 | 134 | 12 | RF + ROS, depth=3 | 96.3% | Looked good but **overfit** — caught by team review |
| v3→v6 | 3.4 + 3.5 + FPDoR cycles | 319 | 39 | RF + SMOTE, depth=10 | 85.4% ± 9.5% | More data → honest AUC; mandatory_pass_rate emerges as #2 signal |

**Key insight:** AUC going *down* from 96.3% → 85.4% is a good sign, not a bad one. With 12 slipped examples and depth=3, the model was memorizing. With 39 slipped examples and depth=10, it's actually learning.

---
## Part A — v1: First Experiment (3.5 only, n=115)

### A1. The imbalance problem

The 3.5 training set had 104 shipped and only **11 slipped** — a 9:1 imbalance. A naive model that always predicts "ship" gets 90% accuracy without learning anything. We tested four strategies to fix this:

| Strategy | Description | Result |
|---|---|---|
| Baseline (`class_weight='balanced'`) | Upweights minority class in loss | LR wins, RF loses |
| **Random Oversampling (ROS)** | Duplicate minority samples until balanced | **RF wins — AUC 91.9%** |
| SMOTE | Synthetic interpolation between minority samples | Hurt RF — only 11 points to interpolate, adds noise |
| ADASYN | Adaptive synthetic sampling | Marginal RF win, not enough to beat ROS |

**Decision: RF + Random Oversampling.** SMOTE failed because interpolating between 11 points creates synthetic negatives that don't reflect real slips — they cluster in the interior of a very small minority manifold.

In [ ]:
# A1: Reproduce the 4-strategy comparison (3.5 only)
# This cell is illustrative — run it if you have the 3.5.json path

import json, numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 42
JIRA_PRI  = {'Critical': 4, 'High': 3, 'Medium': 2, 'Low': 1}
PHASE_ORD = {'EA1': 1, 'EA2': 2, 'GA': 3}

def extract_fpdor(fpdor):
    if not fpdor:
        return dict(pass_rate=0, mandatory_pass_rate=0, criteria_pass_rate=0,
                    passed_count=0, rt_pass=0, docs_pass=0)
    items      = fpdor.get('items', [])
    applicable = [i for i in items if i.get('state') != 'not-checked']
    mandatory  = [i for i in applicable if i.get('group') == 'mandatory']
    criteria   = [i for i in applicable if i.get('group') == 'criteria']
    def pr(lst): return sum(1 for i in lst if i.get('pass')) / len(lst) if lst else 0.0
    rt   = next((i for i in items if i['name'] == 'Release Type'), None)
    docs = next((i for i in items if i['name'] == 'Docs impact'),  None)
    ac   = fpdor.get('applicableCount', 1) or 1
    return dict(
        pass_rate           = fpdor.get('passedCount', 0) / ac,
        mandatory_pass_rate = pr(mandatory),
        criteria_pass_rate  = pr(criteria),
        passed_count        = fpdor.get('passedCount', 0),
        rt_pass             = int(rt['pass'] == True) if rt else 0,
        docs_pass           = int(docs['pass'] == True) if docs else 0,
    )

def load_and_extract(paths):
    rows = []
    for path in paths:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line: rows.append(json.loads(line))
    committed = [r for r in rows if r.get('committedPhase') and r['committedPhase'] not in (None, 'None', 'never')]
    le = LabelEncoder()
    le.fit([r.get('primaryComponent') or 'unknown' for r in committed])
    X, y = [], []
    for r in committed:
        sig = extract_fpdor(r.get('fpdorAtFreeze'))
        rice = r.get('priority', {}).get('rice')
        comp = r.get('primaryComponent') or 'unknown'
        try: comp_enc = le.transform([comp])[0]
        except ValueError: comp_enc = 0
        X.append([
            sig['pass_rate'], sig['mandatory_pass_rate'], sig['criteria_pass_rate'],
            sig['passed_count'], sig['rt_pass'], sig['docs_pass'],
            rice if rice is not None else np.nan,
            JIRA_PRI.get(r.get('priority', {}).get('jiraPriority', ''), 0),
            PHASE_ORD.get(r.get('committedPhase', 'GA'), 3),
            len(r.get('slips') or []),
            int(r.get('hasDocsComponent', False)),
            float(comp_enc),
        ])
        y.append(1 if r.get('deliveredPhase') == r.get('committedPhase') else 0)
    imp = IterativeImputer(max_iter=10, random_state=SEED)
    return imp.fit_transform(np.array(X, dtype=float)), np.array(y)

# Update paths to your local copies
X_v1, y_v1 = load_and_extract(['path/to/3.5.json'])
print(f'v1 dataset: {len(y_v1)} rows  |  {y_v1.sum()} shipped / {(y_v1==0).sum()} slipped')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rf  = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1)
lr  = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)

strategies = [
    ('Baseline (class_weight)',  rf,  None),
    ('LR Baseline',              lr,  None),
    ('RF + Random Oversample',   rf,  RandomOverSampler(random_state=SEED)),
    ('RF + SMOTE',               rf,  SMOTE(random_state=SEED, k_neighbors=3)),
    ('RF + ADASYN',              rf,  ADASYN(random_state=SEED)),
]

results = {}
for name, clf, sampler in strategies:
    if sampler:
        pipe = ImbPipeline([('sampler', sampler), ('clf', clf)])
    else:
        pipe = clf
    scores = cross_val_score(pipe, X_v1, y_v1, cv=skf, scoring='roc_auc')
    results[name] = scores
    print(f'  {name:<35}  AUC={scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%')

In [ ]:
# Plot: 4-strategy comparison
fig, ax = plt.subplots(figsize=(9, 4))
names = list(results.keys())
means = [results[n].mean()*100 for n in names]
stds  = [results[n].std()*100  for n in names]
colors = ['#c62828' if 'LR' in n else '#1565c0' if 'Oversample' in n else '#90caf9' for n in names]
bars = ax.bar(range(len(names)), means, color=colors, alpha=0.85, width=0.6)
ax.errorbar(range(len(names)), means, yerr=stds, fmt='none', color='#333', capsize=5, linewidth=2)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('AUC (%)')
ax.set_ylim(60, 105)
ax.set_title('v1: Balancing Strategy Comparison (3.5 only, n=115)', fontweight='bold')
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 1, f'{m:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.axhline(means[1], color='#c62828', linestyle=':', alpha=0.5, label='LR baseline')
ax.legend()
plt.tight_layout()
plt.savefig('v1_balancing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### A2. v1 result: 100% confidence bug

After wiring RF+ROS into the demo, almost every feature showed 100% confidence. Root cause: **no depth constraint + no CV for depth selection**. The model fit perfectly on the training set and output max probability for unseen features too.

Fix: use cross-validation to select `max_depth` from {3, 5, 7, 10}. With n=115 and 11 slipped, CV picks depth=3 — the shallowest option that generalizes.

---
## Part B — v2: Adding 3.4 data (n=134, AUC 96.3%)

Added RHOAI 3.4 JSONL (114 rows, 8 slipped) to get n=134, 12 slipped total.

Result: depth=3 selected by CV, 100 trees, AUC 96.3%.

**This looked good but was still suspicious.** John Graham flagged it: > 90% AUC on a small dataset with only 12 slipped examples almost always means the model is fitting to noise in those 12 rows. Depth=3 prevents memorization within trees but the forest ensemble can still overfit when k-fold test folds have only 1–2 slipped examples each.

The ±16.7% AUC variance confirmed it: some folds had 0 slipped in the test set (undefined AUC, skipped), and others had 1–2, making the AUC estimate unreliable. We needed more slipped examples — not a better algorithm.

In [ ]:
# B: Reproduce v2 — 3.4 + 3.5, depth=3, 100 trees
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import RandomOverSampler

X_v2, y_v2 = load_and_extract(['path/to/3.4.jsonl', 'path/to/3.5.json'])
print(f'v2 dataset: {len(y_v2)} rows  |  {y_v2.sum()} shipped / {(y_v2==0).sum()} slipped')

pipe_v2 = ImbPipeline([
    ('ros', RandomOverSampler(random_state=SEED)),
    ('rf',  RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1)),
])
grid_v2 = GridSearchCV(
    pipe_v2,
    {'rf__n_estimators': [100], 'rf__max_depth': [3, 5, 7, 10]},
    cv=StratifiedKFold(10, shuffle=True, random_state=SEED),
    scoring='roc_auc', n_jobs=-1,
)
grid_v2.fit(X_v2, y_v2)
print(f'v2 best depth: {grid_v2.best_params_["rf__max_depth"]}')
print(f'v2 CV AUC:     {grid_v2.best_score_*100:.1f}%')
print()
print('⚠️  >90% AUC with 12 slipped examples = red flag.')
print('   AUC variance across folds is high because some folds have 0–1 slipped test samples.')
print('   The model is reliable on the training distribution but not well-generalised.')

---
## Part B3 — The data wall and how we broke it

We needed more slipped examples but had exhausted the obvious sources:
- `3.4.jsonl`: 8 slipped ✓ (already in)
- `3.5.json`: 4 slipped ✓ (already in)
- `features.json`: 16 slipped — **unusable**: no `fpdorAtFreeze` field, and commitment definition conflicts with 3.4.jsonl (different `deliveredPhase` semantics)
- release-capacity-dataset: no outcome labels

**Breakthrough:** The per-phase FPDoR snapshot files (`fpdor-EA1.json`, `fpdor-EA2.json`, `fpdor-GA.json` for both 3.4 and 3.5) have a `closure` field:
- `closed_by_t1` → shipped in that phase
- `done_lag_after_t1` → shipped late but within phase
- `still_open_post_t1` → **slipped**

These give us 185 new labeled rows (158 shipped / 27 slipped) that were not in any feature file. Critically, all 27 new slips have `slip_count=0` — they have never slipped before. This forces the model to learn FPDoR + RICE signals as slip predictors, not just slip history.

See `build_extended_training.py` in this repo for the full extraction pipeline.

---
## Part C — v6: Full Dataset (n=319)

This is the current production model. All sections below run on the full 319-row dataset.

## 1. Setup

In [ ]:
import json, pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss, RocCurveDisplay
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 42
JIRA_PRI  = {'Critical': 4, 'High': 3, 'Medium': 2, 'Low': 1}
PHASE_ORD = {'EA1': 1, 'EA2': 2, 'GA': 3}
print('Libraries loaded.')

## 2. Training Data (v6)

| Source | Rows | Slipped | Notes |
|---|---|---|---|
| `3.4.jsonl` | 114 | 8 | RHOAI 3.4 committed features with full slip history |
| `3.5.json` | 205 | 4 | RHOAI 3.5 committed features |
| `fpdor_cycles_extended.jsonl` | 185 | 27 | Extracted from per-phase FPDoR snapshots — see build_extended_training.py |

In [ ]:
DATA_PATHS = [
    'path/to/3.4.jsonl',           # update to your local path
    'path/to/3.5.json',            # update to your local path
    'fpdor_cycles_extended.jsonl', # in this repo
]

def load_rows():
    rows = []
    for path in DATA_PATHS:
        n = 0
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    r = json.loads(line)
                    r['_source'] = path.split('/')[-1]
                    rows.append(r)
                    n += 1
        print(f'  {n} rows ← {path.split("/")[-1]}')
    return rows

rows = load_rows()
print(f'\nTotal: {len(rows)} rows')

## 3. Feature Extraction

12 features at planning freeze:

| Feature | Source | v1 importance | v6 importance | Why it changed |
|---|---|---|---|---|
| `slip_count` | Prior release history | 52% | 21% | 27 new slips have slip_count=0 — model can't rely on it alone |
| `mandatory_pass_rate` | FPDoR mandatory items | 3% | **19%** | Now sees slips with good vs bad FPDoR — learns the signal |
| `fpdor_pass_rate` | FPDoR overall | 11% | 9% | Stable |
| `rt_pass` | FPDoR: Release Type | 0% | 8% | More examples expose this pattern |
| `jira_priority` | Jira field | — | 8% | Clearer with 3× more data |
| `rice` | RICE score | 12% | 7% | Less dominant with richer FPDoR signal |

In [ ]:
FEATURE_NAMES = [
    'fpdor_pass_rate', 'mandatory_pass_rate', 'criteria_pass_rate',
    'fpdor_passed_count', 'rt_pass', 'docs_pass',
    'rice', 'jira_priority', 'committed_phase_ord',
    'slip_count', 'has_docs_component', 'component_encoded',
]

def build_dataset(rows):
    committed = [r for r in rows
                 if r.get('committedPhase') and r['committedPhase'] not in (None, 'None', 'never')]
    le = LabelEncoder()
    le.fit([r.get('primaryComponent') or 'unknown' for r in committed])
    X_raw, y, keys = [], [], []
    for r in committed:
        label = 1 if r.get('deliveredPhase') == r.get('committedPhase') else 0
        sig   = extract_fpdor(r.get('fpdorAtFreeze'))
        rice  = r.get('priority', {}).get('rice')
        comp  = r.get('primaryComponent') or 'unknown'
        try:    comp_enc = le.transform([comp])[0]
        except  ValueError: comp_enc = 0
        X_raw.append([
            sig['pass_rate'], sig['mandatory_pass_rate'], sig['criteria_pass_rate'],
            sig['passed_count'], sig['rt_pass'], sig['docs_pass'],
            rice if rice is not None else np.nan,
            JIRA_PRI.get(r.get('priority', {}).get('jiraPriority', ''), 0),
            PHASE_ORD.get(r.get('committedPhase', 'GA'), 3),
            len(r.get('slips') or []),
            int(r.get('hasDocsComponent', False)),
            float(comp_enc),
        ])
        y.append(label)
        keys.append(r['key'])
    return np.array(X_raw, dtype=float), np.array(y), keys, le

X_raw, y, keys, comp_le = build_dataset(rows)
n_pos, n_neg = y.sum(), (y == 0).sum()
print(f'Dataset: {len(y)} rows  |  {n_pos} shipped ({n_pos/len(y)*100:.0f}%)  |  {n_neg} slipped ({n_neg/len(y)*100:.0f}%)')

## 4. Dataset Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Class distribution
axes[0].bar(['Shipped', 'Slipped'], [n_pos, n_neg], color=['#2e7d32', '#c62828'], alpha=0.85, width=0.5)
for i, v in enumerate([n_pos, n_neg]):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')
axes[0].set_title('Class Distribution (v6, n=319)', fontweight='bold')
axes[0].set_ylabel('Features'); axes[0].set_ylim(0, 320)

# Source breakdown
sources = Counter(r['_source'] for r in rows)
src_labels = [s.replace('fpdor_cycles_extended.jsonl','FPDoR cycles\n(3.4+3.5)')
              .replace('3.4.jsonl','3.4').replace('3.5.json','3.5') for s in sources]
axes[1].bar(src_labels, sources.values(), color=['#1565c0','#1976d2','#42a5f5'], alpha=0.85, width=0.5)
for i, v in enumerate(sources.values()):
    axes[1].text(i, v+1, str(v), ha='center', fontweight='bold')
axes[1].set_title('Rows by Source', fontweight='bold')
axes[1].set_ylabel('Rows')

# Version history: n and AUC
versions = ['v1\n(3.5 only)', 'v2\n(3.4+3.5)', 'v6\n(+FPDoR cycles)']
ns       = [115, 134, 319]
aucs_h   = [91.9, 96.3, 85.4]
ax2 = axes[2].twinx()
axes[2].bar(versions, ns, color='#90caf9', alpha=0.7, width=0.4, label='n (rows)')
ax2.plot(versions, aucs_h, 'o-', color='#c62828', linewidth=2, markersize=8, label='AUC %')
axes[2].set_ylabel('Training rows'); ax2.set_ylabel('AUC (%)')
ax2.set_ylim(70, 105)
axes[2].set_title('Version history: data growth vs AUC', fontweight='bold')
axes[2].legend(loc='upper left'); ax2.legend(loc='upper right')
for i, (n, a) in enumerate(zip(ns, aucs_h)):
    ax2.text(i, a+1, f'{a}%', ha='center', fontsize=9, color='#c62828', fontweight='bold')

plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: AUC drop v2→v6 is intentional — more data exposed the overfitting.')

## 5. Imputation (MICE)

`rice` is missing for some features. MICE (Multiple Imputation by Chained Equations) fills NaNs using the joint distribution of all other features — better than median because it uses real correlations.

In [ ]:
mice_imp = IterativeImputer(max_iter=10, random_state=SEED, initial_strategy='median')
X_clean  = mice_imp.fit_transform(X_raw)
print(f'NaN values: {np.isnan(X_raw).sum()} → {np.isnan(X_clean).sum()} (MICE imputation)')

## 6. Hyperparameter Tuning

GridSearchCV with 10-fold stratified CV. SMOTE is inside the pipeline — it only sees training folds, never leaks into the test fold.

In [ ]:
def make_rf(**kwargs):
    return RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1, **kwargs)

pipe = ImbPipeline([
    ('smote', SMOTE(random_state=SEED, k_neighbors=5)),
    ('rf',    make_rf()),
])
param_grid = {
    'rf__n_estimators':     [100, 200, 300],
    'rf__max_depth':        [3, 5, 7, 10],
    'rf__min_samples_leaf': [3, 5, 10],
}
grid = GridSearchCV(
    estimator=pipe, param_grid=param_grid,
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED),
    scoring='roc_auc', n_jobs=-1,
)
grid.fit(X_clean, y)
best_params = {k.replace('rf__', ''): v for k, v in grid.best_params_.items()}
print(f'Best params: {best_params}')
print(f'Best CV AUC: {grid.best_score_*100:.1f}%')
print()
print('Note: with n=319, CV now selects depth=10 (not depth=3 as in v2).')
print('The model can go deeper without memorizing because it has 3× more data.')

## 7. Cross-Validation (10-fold)

Each fold: ~287 train / ~32 test. SMOTE applied to training fold only.

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
smote = SMOTE(random_state=SEED, k_neighbors=5)
aucs, briers, f1s = [], [], []

for fold, (tr_idx, te_idx) in enumerate(skf.split(X_clean, y), 1):
    X_tr, X_te = X_clean[tr_idx], X_clean[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    if len(np.unique(y_te)) < 2:
        print(f'  Fold {fold}: skipped (single class in test)')
        continue
    X_tr_b, y_tr_b = smote.fit_resample(X_tr, y_tr)
    rf = make_rf(**best_params)
    rf.fit(X_tr_b, y_tr_b)
    probs = rf.predict_proba(X_te)[:, 1]
    preds = rf.predict(X_te)
    aucs.append(roc_auc_score(y_te, probs))
    briers.append(brier_score_loss(y_te, probs))
    f1s.append(f1_score(y_te, preds, zero_division=0))
    print(f'  Fold {fold}: AUC={aucs[-1]*100:.1f}%  Brier={briers[-1]:.3f}  F1={f1s[-1]*100:.1f}%')

print(f'\nMean AUC:   {np.mean(aucs)*100:.1f}% ± {np.std(aucs)*100:.1f}%')
print(f'Mean Brier: {np.mean(briers):.3f}')
print(f'Mean F1:    {np.mean(f1s)*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(range(1, len(aucs)+1), [a*100 for a in aucs], color='#1565c0', alpha=0.8)
axes[0].axhline(np.mean(aucs)*100, color='#c62828', linewidth=2, linestyle='--',
                label=f'Mean {np.mean(aucs)*100:.1f}% ± {np.std(aucs)*100:.1f}%')
axes[0].set_xlabel('Fold'); axes[0].set_ylabel('AUC (%)')
axes[0].set_title('AUC per CV Fold (v6)', fontweight='bold')
axes[0].set_ylim(50, 105); axes[0].legend()

probs_cv = cross_val_predict(
    make_rf(**best_params), X_clean, y,
    cv=StratifiedKFold(10, shuffle=True, random_state=SEED),
    method='predict_proba'
)[:, 1]
RocCurveDisplay.from_predictions(y, probs_cv, ax=axes[1], color='#1565c0')
axes[1].set_title('ROC Curve (10-fold CV)', fontweight='bold')
axes[1].plot([0,1],[0,1],'k--',alpha=0.4)
plt.tight_layout()
plt.savefig('cv_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Final Model + Calibration

In [ ]:
X_b, y_b = smote.fit_resample(X_clean, y)
print(f'After SMOTE: {Counter(y_b)}')

rf_final = make_rf(**best_params)
rf_final.fit(X_b, y_b)

rf_calibrated = CalibratedClassifierCV(make_rf(**best_params), cv=5, method='isotonic')
rf_calibrated.fit(X_b, y_b)
print('Model trained and calibrated.')

## 9. Calibration Check

In [ ]:
probs_cal = cross_val_predict(
    CalibratedClassifierCV(make_rf(**best_params), cv=5, method='isotonic'),
    X_clean, y,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    method='predict_proba',
)[:, 1]

frac_pos, mean_pred = calibration_curve(y, probs_cal, n_bins=5)
ece = np.sum(np.abs(frac_pos - mean_pred) * (len(y) / 5)) / len(y)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(mean_pred, frac_pos, 'o-', color='#1565c0', label='Calibrated RF', linewidth=2, markersize=8)
ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives (actual ship rate)')
ax.set_title(f'Reliability Diagram  (ECE = {ece:.3f})', fontweight='bold')
ax.legend(); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig('calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'ECE = {ece:.3f}  (target < 0.05; currently {ece:.3f} — improving with more data)')

## 10. Feature Importance

In [ ]:
importances = rf_final.feature_importances_
order = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1565c0' if importances[i] >= 0.10 else '#42a5f5' if importances[i] >= 0.05 else '#90caf9'
          for i in order]
ax.barh([FEATURE_NAMES[i] for i in order[::-1]],
        [importances[i]*100 for i in order[::-1]],
        color=colors[::-1], alpha=0.85)
ax.set_xlabel('Importance (%)')
ax.set_title('Feature Importance — v6 (depth=10, n=319)', fontweight='bold')
for i, imp in enumerate([importances[i] for i in order[::-1]]):
    ax.text(imp*100+0.2, i, f'{imp*100:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Save Model

In [ ]:
out = {
    'rf':            rf_final,
    'rf_calibrated': rf_calibrated,
    'imputer':       mice_imp,
    'comp_le':       comp_le,
    'feature_names': FEATURE_NAMES,
    'best_params':   best_params,
    'cv_metrics':    {'auc': aucs, 'brier': briers, 'f1': f1s},
}
with open('models_v3.pkl', 'wb') as f:
    pickle.dump(out, f)
print('Saved → models_v3.pkl')
print(f'  Training set : {len(y)} rows ({n_pos} shipped / {n_neg} slipped)')
print(f'  Best params  : {best_params}')
print(f'  CV AUC       : {np.mean(aucs)*100:.1f}% ± {np.std(aucs)*100:.1f}%')
print(f'  Brier        : {np.mean(briers):.3f}')
print(f'  ECE          : {ece:.3f}')

## 12. What's Next

| Path | Expected impact | Status |
|---|---|---|
| 3.6 ships → labeled rows | n → ~950, AUC variance ±9.5% → ±4–5% | Automatic at GA |
| Jira API via org-pulse bot account | Real `slip_count` for 568 ~🤖 features → drop 85% cap | Pending bot account |
| FPDoR items alignment with Erle | Ensures training features match live scoring | In progress |
| SHAP per-feature explanations | Per-feature "why" in REASON column | Not yet built |

---
*Demo: https://github.com/yuvalluria/rhoai-release-planner — open `index.html`, no server needed.*